# Hybrid: Mult-VAE (collaborative) ⊕ max-sim (content) — UC1
Fuses the trained Mult-VAE with the content/max-sim scorer via Reciprocal Rank Fusion, sweeps the content weight, and splits by history length to show *why* content helps the tail.

**Kaggle inputs required:** `sample.parquet`, `multvae_last.pt`, `embeddings.npy`, `catalog.parquet`.

In [ ]:
import os
REPO = '/kaggle/working/book_recsys'
if not os.path.exists(REPO):
    !git clone -b worktree-autoencoder https://github.com/MayaDeneva/book_recsys.git {REPO}
else:
    !cd {REPO} && git pull origin worktree-autoencoder
%cd {REPO}
import sys; sys.path.insert(0, REPO)
import book_recsys.models.autoencoder.recommender  # smoke
print('ready')

In [ ]:
import glob, numpy as np, pandas as pd
from book_recsys.data.schema import BOOK
from book_recsys.data.splits import leave_last_n_out
from book_recsys.models.autoencoder.data import reproduce_sasrec_sample
from book_recsys.eval.harness import build_user_histories, build_relevance

DEVICE, N_EVAL_USERS = 'cpu', 2000   # CPU notebook (GPU doesn't help the NumPy max-sim)
sample = reproduce_sasrec_sample(
    pd.read_parquet(glob.glob('/kaggle/input/**/sample.parquet', recursive=True)[0]),
    30000, 100, expect_rows=None)
train, test = leave_last_n_out(sample, n=1)
histories = build_user_histories(train)
relevance = build_relevance(test)
relevance = {u: relevance[u] for u in list(relevance)[:N_EVAL_USERS]}
all_items = sample[BOOK].unique()
weights = sample[BOOK].value_counts().reindex(all_items).to_numpy()
print(len(relevance), 'eval users')

In [ ]:
from book_recsys.models.autoencoder.train import load_checkpoint
from book_recsys.models.autoencoder.recommender import MultVaeRecommender

model, ckpt = load_checkpoint(glob.glob('/kaggle/input/**/multvae_last.pt', recursive=True)[0], device=DEVICE)
ids = ckpt['ids']; pos = {b: j for j, b in enumerate(ids)}
counts = train[BOOK].value_counts().reindex(ids).to_numpy(dtype=float)
rec = MultVaeRecommender(device=DEVICE).attach(model, ids, pos, counts)
print('vae epoch', ckpt['epoch'], '| on', next(rec._model.parameters()).device)

In [ ]:
from book_recsys.models.content.maxsim import MaxSimRecommender
embeddings = np.load(glob.glob('/kaggle/input/**/embeddings.npy', recursive=True)[0])
catalog = pd.read_parquet(glob.glob('/kaggle/input/**/catalog.parquet', recursive=True)[0])
rec_max = MaxSimRecommender(catalog['book_id'].tolist(), embeddings)  # rows row-aligned to catalog
print('maxsim catalog:', len(catalog), '| emb dim:', embeddings.shape[1])

## Fusion weight sweep
`maxsim_weight=0` is pure Mult-VAE; ↑ = more content. Watch NDCG/recall.

In [ ]:
from book_recsys.models.ensemble import RRFEnsembleRecommender
from book_recsys.eval.harness import evaluate_sampled_negatives

rows = []
for w_max in (0.0, 0.5, 1.0, 2.0, 4.0):
    ens = RRFEnsembleRecommender({'vae': rec, 'maxsim': rec_max},
                                 weights={'vae': 1.0, 'maxsim': w_max})
    m = evaluate_sampled_negatives(ens, histories, relevance, all_items,
                                   n_neg=100, k=10, seed=0, item_weights=weights)
    rows.append({'maxsim_weight': w_max, **m}); print('done', w_max)
pd.DataFrame(rows)

## Tail / coverage at the chosen weight (subset — full-catalog max-sim ranking is heavy)

In [ ]:
from book_recsys.eval.harness import popularity_diagnostics
W = 1.0  # set to your best row above
ens = RRFEnsembleRecommender({'vae': rec, 'maxsim': rec_max}, weights={'vae': 1.0, 'maxsim': W})
sub = {u: histories.get(u, []) for u in list(relevance)[:400]}
print(popularity_diagnostics(ens, sub, list(sample[BOOK].value_counts().index),
                             catalog_size=len(all_items), k=10))

## Short- vs long-history users
k-core (users ≥20 interactions) removed true cold-start users, so this is a **short- vs long-history** split at the median. Expectation: content/hybrid helps short-history users (little collaborative signal) more than pure CF.

In [ ]:
from book_recsys.eval.harness import cold_warm_users
thr = int(np.median([len(histories.get(u, [])) for u in relevance]))
short, long_ = cold_warm_users({u: histories.get(u, []) for u in relevance}, threshold=thr)
print(f'median history={thr} | short={len(short)} long={len(long_)}')

hybrid = RRFEnsembleRecommender({'vae': rec, 'maxsim': rec_max}, weights={'vae': 1.0, 'maxsim': 1.0})
models = {'mult-vae': rec, 'maxsim': rec_max, 'hybrid': hybrid}
rows = []
for name, mdl in models.items():
    for seg, users in [('short', short), ('long', long_)]:
        rel = {u: relevance[u] for u in relevance if u in users}
        r = evaluate_sampled_negatives(mdl, histories, rel, all_items,
                                       n_neg=100, k=10, seed=0, item_weights=weights)
        rows.append({'model': name, 'segment': seg, 'n_users': len(rel), **r})
pd.DataFrame(rows)

## Tail diagnostics per model (for the comparison table)
Coverage + mean-pop-percentile for Mult-VAE (α=1), max-sim, and hybrid on a 400-user subset (full-catalog ranking with max-sim is heavy). Lower pop-percentile + higher coverage = less bestseller-biased.

In [ ]:
from book_recsys.eval.harness import popularity_diagnostics
rec.pop_discount = 1.0          # lock Mult-VAE at its best alpha
HYB_W = 1.0                     # set to your best fusion weight from the sweep above
hybrid = RRFEnsembleRecommender({'vae': rec, 'maxsim': rec_max},
                                weights={'vae': 1.0, 'maxsim': HYB_W})
pop_order = list(sample[BOOK].value_counts().index)
sub = {u: histories.get(u, []) for u in list(relevance)[:400]}
rows = []
for name, mdl in {'mult-vae (a=1)': rec, 'maxsim': rec_max, 'hybrid': hybrid}.items():
    d = popularity_diagnostics(mdl, sub, pop_order, catalog_size=len(all_items), k=10)
    rows.append({'model': name, **d}); print('done', name)
pd.DataFrame(rows)